<a href="https://colab.research.google.com/github/JediPingu/Particle-Physics-and-ML/blob/main/Higgs_Boson_Challenge_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn, optim
import pandas as pd
import numpy as np
import math

In [ ]:
df = pd.read_csv("training.csv")
df["Label"] = (df["Label"] == "s").astype(int)

# Model 1 : Random forest

In [ ]:
import sklearn as skl
from sklearn.ensemble import RandomForestClassifier

In [ ]:
cols = ["DER_mass_MMC","DER_mass_transverse_met_lep","DER_mass_vis","DER_pt_h","DER_deltaeta_jet_jet","DER_mass_jet_jet","DER_prodeta_jet_jet","DER_deltar_tau_lep","DER_pt_tot","DER_sum_pt","DER_pt_ratio_lep_tau","DER_met_phi_centrality","DER_lep_eta_centrality","PRI_tau_pt","PRI_tau_eta","PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt","PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt","Weight"]
inputs = df.loc[:, cols]
y = df.loc[:, "Label"]

input_train, input_test, y_train, y_test = skl.model_selection.train_test_split(inputs, y, test_size=170000)

model = RandomForestClassifier(n_estimators=32)
model.fit(input_train, y_train)
print(model.score(input_test, y_test))

1.0


# Model 2 : Multilayer perceptron

In [ ]:
class HiggsDataSet(torch.utils.data.Dataset):
    def __init__(self, df):
        self.data = df
        # Get columns of input data, to exclude the final label
        self.columns = ["DER_mass_MMC","DER_mass_transverse_met_lep","DER_mass_vis","DER_pt_h","DER_deltaeta_jet_jet","DER_mass_jet_jet","DER_prodeta_jet_jet","DER_deltar_tau_lep","DER_pt_tot","DER_sum_pt","DER_pt_ratio_lep_tau","DER_met_phi_centrality","DER_lep_eta_centrality","PRI_tau_pt","PRI_tau_eta","PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt","PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt","Weight"]

    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        # Return a tuple containing the input values as the first element, and the label to compare against as the second element
        return torch.tensor([float(self.data[col][idx]) for col in self.columns]), torch.tensor(float(self.data["Label"][idx]))


dataset = HiggsDataSet(df)
print(dataset[2][0].shape)
BATCH_SIZE = 64
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

torch.Size([31])


In [ ]:
class HiggsBosonClassifier(nn.Module):
    def __init__(self):
        super(HiggsBosonClassifier, self).__init__()
        self.fc1 = nn.Linear(31, 64)
        self.norm1 = nn.LayerNorm(64)

        self.fc2 = nn.Linear(64, 32)
        self.norm2 = nn.LayerNorm(64)

        self.fc3 = nn.Linear(32, 1)
        self.norm3 = nn.LayerNorm(256)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.dropout(self.relu(self.norm1(self.fc1(x))))
        x = self.dropout(self.relu(self.norm2(self.fc2(x))))
        x = self.dropout(self.relu(self.norm3(self.fc3(x))))
        x = self.sigmoid(x)
        return x
model = HiggsBosonClassifier()
optimiser = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
loss_fn = nn.BCELoss()
loss_fn(model(dataset[0][0]), dataset[0][1].unsqueeze(-1))

tensor(0.7169, grad_fn=<BinaryCrossEntropyBackward0>)

In [ ]:
def training_loop(epochs: int):
    for epoch in range(epochs):
        epoch_loss = 0
        correct = 0
        for i, batch in enumerate(loader):
            output = model(batch[0])
            try:
                loss = loss_fn(output, batch[1].unsqueeze(-1))
            except RuntimeError:
                print(output)
                print(batch[1])

            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
            if i % 500 == 499:
                print(f"Batch {i+1}/{int(len(dataset)/BATCH_SIZE)} | Loss : {loss.item()}")
            for t in range(len(output)):
                if torch.round(output[t]) == batch[1][t]:
                    correct += 1
        print(f"Epoch {epoch+1} | Loss : {(epoch_loss*BATCH_SIZE)/len(dataset)} | Accuracy : {correct/len(dataset)}")
training_loop(5)
